# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all `RecordSet`s (`@id`s) in the dataset and preview their included fields (`@id`s).

In [ ]:
# Retrieve all RecordSet @id's
record_sets = dataset.get_record_sets()
print('Available RecordSets:')
for rs in record_sets:
    print(f"  - {rs['@id']} ({rs['name'] if 'name' in rs else 'No name'})")
    # List fields for each RecordSet by @id
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("    Fields:")
        for f in fields:
            print(f"      - {f['@id'] if isinstance(f, dict) and '@id' in f else str(f)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

---
**Example below uses the first available RecordSet. Adjust as needed for your analysis.**

In [ ]:
# Extract data from each record set
from collections.abc import Iterable

# Get all RecordSet @id's
record_set_ids = [rs['@id'] for rs in dataset.get_record_sets()]
dataframes = {}

# If no record sets found, raise error
if not record_set_ids:
    raise RuntimeError('No RecordSets found in metadata.')

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    # Convert to list and DataFrame
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  {len(df)} records loaded. Columns: {df.columns.tolist()}")

# Example: Show columns for the first record set and preview data
main_record_set_id = record_set_ids[0]
print(f"Columns for RecordSet {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

---
**Below, replace `numeric_field_id` and `group_field_id` with actual field `@id`s from the dataset.**

In [ ]:
# Select a numeric field for analysis
df = dataframes[main_record_set_id]

# List numeric fields
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    raise RuntimeError('No numeric fields found in first RecordSet.')
# Use the first numeric field for demonstration
numeric_field_id = numeric_fields[0]

threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a field if there are categorical fields
categorical_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
if categorical_fields:
    group_field_id = categorical_fields[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If there are at least two numeric fields, plot scatter
if len(numeric_fields) >= 2:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.title(f'Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains one or more record sets, each corresponding to a different aspect of mass spectrometry analysis of extracellular vesicles.
- Numerically rich fields (such as abundance, score, or molecular weight, as demonstrated by the first numeric column analyzed) can be extracted, normalized, and visualized easily using `mlcroissant` with their field `@id`.
- Use the field and record set `@id`s for reference to ensure you are extracting and manipulating the correct data columns.
- The filtering, normalization, and groupings shown here can be adapted to the dataset’s structure based on scientific needs and questions.

For further analysis, refer to the detailed `@id`s and consider visualizing or aggregating according to experimental groupings (e.g., sample types, peptide categories, etc).